# Bước 1 — Chuẩn bị input cho Phase 2 Sentiment

Notebook tạo bảng bài báo đầu vào cho Phase 2. Mỗi `news_id` là một dòng và không tách theo ticker.

- **Input:** `news_processing/data/processed/news_type_predictions.csv`
- **Output:** `news_processing/phase_2_sentiment/data/sentiment_input.csv`
- Loại 4 bài đã được Phase 1 đánh dấu `excluded`.
- Không sử dụng dữ liệu hoặc nhãn Sentiment cũ.
- Chưa tạo cột nhãn, chưa chia dữ liệu và chưa train model.

## 1. Thư viện và đường dẫn

In [29]:
from __future__ import annotations

import hashlib
import os
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

In [30]:
current_working_directory = Path.cwd().resolve()
phase_directory_candidates = [
    current_working_directory / 'news_processing' / 'phase_2_sentiment',
    current_working_directory / 'phase_2_sentiment',
    current_working_directory,
    *current_working_directory.parents,
]
phase_directory = next(
    (candidate for candidate in phase_directory_candidates
     if candidate.name == 'phase_2_sentiment'
     and (candidate / 'notebooks').is_dir()),
    None,
)
if phase_directory is None:
    raise FileNotFoundError('Không tìm thấy news_processing/phase_2_sentiment.')

news_processing_directory = phase_directory.parent
phase_one_input_path = news_processing_directory / 'data/processed/news_type_predictions.csv'
phase_two_output_path = phase_directory / 'data/sentiment_input.csv'

print('Phase directory:', phase_directory)
print('Input:', phase_one_input_path)
print('Output:', phase_two_output_path)

Phase directory: D:\stock_news_anomaly_project\news_processing\phase_2_sentiment
Input: D:\stock_news_anomaly_project\news_processing\data\processed\news_type_predictions.csv
Output: D:\stock_news_anomaly_project\news_processing\phase_2_sentiment\data\sentiment_input.csv


## 2. Cấu hình dữ liệu đã khóa

In [31]:
expected_input_sha256 = '15d73d1cf809d6c7cd2708584814e526006648fa68b07b316f0dc449995021d8'
expected_total_articles = 3354
expected_eligible_articles = 3350
expected_excluded_articles = 4
expected_label_source_counts = {'manual': 1090, 'model': 2260, 'excluded': 4}
csv_encoding = 'utf-8-sig'

phase_one_columns = [
    'news_id', 'source', 'url', 'title', 'summary', 'content',
    'published_date', 'tickers', 'news_type', 'label_source',
    'prediction_score',
]
phase_two_input_columns = [
    'news_id', 'source', 'url', 'title', 'summary', 'content',
    'published_date', 'tickers', 'news_type',
]

In [32]:
def ensure_condition(condition: bool, error_message: str) -> None:
    if not condition:
        raise ValueError(error_message)


def calculate_sha256(file_path: Path) -> str:
    return hashlib.sha256(file_path.read_bytes()).hexdigest()

## 3. Đọc dữ liệu Phase 1

In [33]:
ensure_condition(phase_one_input_path.is_file(), f'Không tìm thấy input: {phase_one_input_path}')
phase_one_news = pd.read_csv(
    phase_one_input_path,
    dtype=str,
    keep_default_na=False,
)
phase_one_input_sha256 = calculate_sha256(phase_one_input_path)

print('Số bài đọc từ Phase 1:', len(phase_one_news))
print('SHA-256:', phase_one_input_sha256)

Số bài đọc từ Phase 1: 3354
SHA-256: 15d73d1cf809d6c7cd2708584814e526006648fa68b07b316f0dc449995021d8


## 4. Kiểm tra hợp đồng dữ liệu Phase 1

In [34]:
ensure_condition(
    phase_one_input_sha256 == expected_input_sha256,
    f'Checksum input không khớp: {phase_one_input_sha256}',
)
ensure_condition(
    phase_one_news.columns.tolist() == phase_one_columns,
    'Schema input không đúng bản Phase 1 đã khóa.',
)
ensure_condition(
    len(phase_one_news) == expected_total_articles,
    f'Input phải có {expected_total_articles} bài.',
)
ensure_condition(not phase_one_news['news_id'].duplicated().any(), 'Input có news_id trùng.')
ensure_condition(not phase_one_news['news_id'].str.strip().eq('').any(), 'Input có news_id trống.')

actual_label_source_counts = phase_one_news['label_source'].value_counts().to_dict()
ensure_condition(
    actual_label_source_counts == expected_label_source_counts,
    f'Phân bố label_source sai: {actual_label_source_counts}',
)

source_contract_audit = pd.DataFrame([
    {'check': 'Checksum Phase 1', 'expected': expected_input_sha256, 'actual': phase_one_input_sha256, 'result': 'PASS'},
    {'check': 'Tổng số bài', 'expected': expected_total_articles, 'actual': len(phase_one_news), 'result': 'PASS'},
    {'check': 'news_id trùng', 'expected': 0, 'actual': int(phase_one_news['news_id'].duplicated().sum()), 'result': 'PASS'},
])
display(source_contract_audit)

,check,expected,actual,result
0,Checksum Phase 1,15d73d1cf809d6c7cd2708584814e526006648fa68b07b...,15d73d1cf809d6c7cd2708584814e526006648fa68b07b...,PASS
1,Tổng số bài,3354,3354,PASS
2,news_id trùng,0,0,PASS


## 5. Loại bài excluded

In [35]:
excluded_article_mask = phase_one_news['label_source'].eq('excluded')
excluded_news = phase_one_news.loc[excluded_article_mask].copy().reset_index(drop=True)
eligible_news = phase_one_news.loc[~excluded_article_mask].copy().reset_index(drop=True)

ensure_condition(len(excluded_news) == expected_excluded_articles, 'Sai số bài excluded.')
ensure_condition(len(eligible_news) == expected_eligible_articles, 'Sai số bài hợp lệ.')
ensure_condition(excluded_news['news_type'].str.strip().eq('').all(), 'Bài excluded không được có news_type.')
ensure_condition(excluded_news['prediction_score'].str.strip().eq('').all(), 'Bài excluded không được có prediction_score.')
ensure_condition(eligible_news['news_id'].nunique() == expected_eligible_articles, 'news_id hợp lệ không duy nhất.')

for column_name in phase_two_input_columns:
    ensure_condition(
        not eligible_news[column_name].str.strip().eq('').any(),
        f'Bài hợp lệ có giá trị trống tại cột {column_name}.',
    )

print('Bài hợp lệ:', len(eligible_news))
print('Bài excluded:', len(excluded_news))

Bài hợp lệ: 3350
Bài excluded: 4


## 6. Tạo bảng input Phase 2

In [36]:
sentiment_input = eligible_news[phase_two_input_columns].copy().reset_index(drop=True)

ensure_condition(len(sentiment_input) == expected_eligible_articles, 'Sai số dòng output.')
ensure_condition(sentiment_input['news_id'].nunique() == expected_eligible_articles, 'Output có news_id trùng.')
ensure_condition('sentiment' not in sentiment_input.columns, 'Bước 1 chưa được có cột sentiment.')

print('Kích thước bảng input:', sentiment_input.shape)

Kích thước bảng input: (3350, 9)


In [37]:
preview_columns = ['news_id', 'title', 'tickers', 'news_type']
display(sentiment_input[preview_columns].head(5))

,news_id,title,tickers,news_type
0,57a3bac8e0b55bd495bf688a17e6a9f3a3b6fef06373ae...,"Masan liên tiếp khai trương cửa hàng CVLife, t...",MSN,business_operations
1,9cddfe019c634b81361fb00a77a9bae5bbc61795e00ce8...,Liên doanh Del Monte-Vinamilk tại Philippines ...,VNM,business_operations
2,e1751ac1f9ba5c069ce22bae353e052684d1f59c8d23b3...,Vinamilk ủng hộ 10 tỷ đồng và 1 triệu ly sữa c...,VNM,business_operations
3,f97a7efa668f1e2d51930ff67d119c64177b3209afd7e6...,Vinamilk hỗ trợ 10 tỷ đồng và 1 triệu ly sữa c...,VNM,business_operations
4,bfc140797b8dca4f01f53cc8125ed58a1c172ef49303b3...,PNJ lỗ tháng thứ 3 liên tiếp,PNJ,financial_results


## 7. Kiểm tra output hiện có

In [38]:
def read_and_validate_sentiment_input(file_path: Path) -> pd.DataFrame:
    exported_table = pd.read_csv(
        file_path,
        dtype=str,
        keep_default_na=False,
        encoding=csv_encoding,
    )
    ensure_condition(exported_table.columns.tolist() == phase_two_input_columns, 'CSV output sai schema.')
    ensure_condition(len(exported_table) == expected_eligible_articles, 'CSV output sai số dòng.')
    ensure_condition(exported_table['news_id'].nunique() == expected_eligible_articles, 'CSV output có news_id trùng.')
    ensure_condition('sentiment' not in exported_table.columns, 'CSV Step 1 không được có cột sentiment.')
    ensure_condition(
        not exported_table['news_id'].isin(set(excluded_news['news_id'])).any(),
        'CSV còn chứa bài excluded.',
    )
    pd.testing.assert_frame_equal(exported_table, sentiment_input, check_dtype=False)
    return exported_table

In [39]:
if phase_two_output_path.is_file():
    read_and_validate_sentiment_input(phase_two_output_path)
    print('Output CSV hiện có khớp dữ liệu nguồn: cho phép tái tạo.')
else:
    print('Output CSV chưa tồn tại: sẽ tạo mới.')

Output CSV hiện có khớp dữ liệu nguồn: cho phép tái tạo.


## 8. Ghi CSV

In [40]:
def write_sentiment_input_csv(output_table: pd.DataFrame, file_path: Path) -> None:
    output_table.to_csv(
        file_path,
        index=False,
        encoding=csv_encoding,
        lineterminator='\n',
    )

In [41]:
phase_two_output_path.parent.mkdir(parents=True, exist_ok=True)
temporary_file_descriptor, temporary_file_name = tempfile.mkstemp(
    prefix='sentiment_input_',
    suffix='.csv',
    dir=phase_two_output_path.parent,
)
os.close(temporary_file_descriptor)
temporary_output_path = Path(temporary_file_name)

try:
    write_sentiment_input_csv(sentiment_input, temporary_output_path)
    read_and_validate_sentiment_input(temporary_output_path)
    os.replace(temporary_output_path, phase_two_output_path)
finally:
    if temporary_output_path.exists():
        temporary_output_path.unlink()

print('Đã ghi output:', phase_two_output_path)

Đã ghi output: D:\stock_news_anomaly_project\news_processing\phase_2_sentiment\data\sentiment_input.csv


## 9. Nghiệm thu Bước 1

In [42]:
verified_sentiment_input = read_and_validate_sentiment_input(phase_two_output_path)
output_sha256 = calculate_sha256(phase_two_output_path)
temporary_files_remaining = list(phase_two_output_path.parent.glob('sentiment_input_*.csv'))
ensure_condition(not temporary_files_remaining, 'Còn file CSV tạm sau khi xuất.')

acceptance_results = pd.DataFrame([
    {'check': 'Số bài output', 'expected': expected_eligible_articles, 'actual': len(verified_sentiment_input), 'result': 'PASS'},
    {'check': 'Số cột output', 'expected': len(phase_two_input_columns), 'actual': len(verified_sentiment_input.columns), 'result': 'PASS'},
    {'check': 'news_id duy nhất', 'expected': expected_eligible_articles, 'actual': verified_sentiment_input['news_id'].nunique(), 'result': 'PASS'},
    {'check': 'Bài excluded trong output', 'expected': 0, 'actual': int(verified_sentiment_input['news_id'].isin(set(excluded_news['news_id'])).sum()), 'result': 'PASS'},
    {'check': 'Cột nhãn trong Step 1', 'expected': 0, 'actual': int('sentiment' in verified_sentiment_input.columns), 'result': 'PASS'},
])
display(acceptance_results)
print('STEP 1: PASS')
print('Output SHA-256:', output_sha256)

,check,expected,actual,result
0,Số bài output,3350,3350,PASS
1,Số cột output,9,9,PASS
2,news_id duy nhất,3350,3350,PASS
3,Bài excluded trong output,0,0,PASS
4,Cột nhãn trong Step 1,0,0,PASS


STEP 1: PASS
Output SHA-256: c281a5bfc74a80fc20c689df24c7a02f4bec75cde07e622b4c2440e15067c0aa


## Kết quả

Bước 1 tạo file CSV gồm 3.350 bài và 9 cột dữ liệu nguồn. Cột nhãn chỉ được tạo ở bước gán nhãn sau khi quy tắc `NEG/NEU/POS` được chốt.